In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from plotly.offline import iplot
from statsmodels.tsa.stattools import adfuller
from utils import globals
import os

In [2]:
train = None
with open("../../Data/PHM2025_training_data/training_data.csv", "r") as f:
    train = pd.read_csv(f)

train.head()

,ESN,Cycles_Since_New,Snapshot,Cumulative_WWs,Cumulative_HPC_SVs,Cumulative_HPT_SVs,Sensed_Altitude,Sensed_Mach,Sensed_Pamb,Sensed_Pt2,...,Sensed_Core_Speed,Sensed_T25,Sensed_T3,Sensed_Ps3,Sensed_T45,Sensed_P25,Sensed_T5,Cycles_to_WW,Cycles_to_HPC_SV,Cycles_to_HPT_SV
0,101,0,1,0,0,0,851.321516,0.103224,14.250475,14.289459,...,21261.365607,744.009196,1609.417021,474.158862,2173.316575,40.065810,1415.477470,910,8510,4090
1,101,0,2,0,0,0,903.321516,0.213663,14.224853,14.628042,...,21280.040466,747.318490,1612.148051,483.190299,2172.105766,41.152081,1409.643967,910,8510,4090
2,101,0,3,0,0,0,3410.321516,0.345333,12.989563,14.055733,...,21014.282876,735.730180,1576.031353,442.421527,2106.463686,38.758249,1365.412495,910,8510,4090
3,101,0,4,0,0,0,20259.321516,0.619005,6.679796,8.621275,...,20466.466115,706.293618,1504.811122,283.988928,2016.969960,25.574967,1256.689540,910,8510,4090
4,101,0,6,0,0,0,21991.321516,0.640425,6.230955,8.183755,...,20313.501657,699.650483,1484.629857,261.543915,1984.370026,23.927579,1238.665408,910,8510,4090


In [3]:
# Add a cycle index 'scaled_index' for each ESN
train['scaled_index'] = train.groupby('ESN').cumcount()

In [4]:
# Identificazione degli event points
# Assuming your DataFrame is 'df' and the column is 'sensor_value'
wws_points = train[train['Cumulative_WWs'] > train['Cumulative_WWs'].shift(1)]
hpc_points = train[train['Cumulative_HPC_SVs'] > train['Cumulative_HPC_SVs'].shift(1)]
hpt_points = train[train['Cumulative_HPT_SVs'] > train['Cumulative_HPT_SVs'].shift(1)]

In [ ]:
# Test per verificare la stazionarietà dei segnali dei sensori per Snapshot - controllo delle statistiche mobili dei segnali

output_dir = globals.PLOT_PATH + "/STAIONARITY-BY-SNAPSHOT"
os.makedirs(output_dir, exist_ok=True)

for esn_id in globals.ESN:
    sensor_data_tot = train[train['ESN'] == esn_id].copy()
    for snapshot in globals.Snapshot:
        sensor_data = sensor_data_tot[sensor_data_tot['Snapshot'] == snapshot].copy() 
        # Se uno snapshot ha troppi pochi dati, lo salta
        if len(sensor_data) < 5:
            continue
        fig, axes = plt.subplots(4, 4, figsize=(20, 16))
        axes = axes.flatten()
        print(f"Generazione dashboard per Motore ESN: {esn_id} - Snapshot: {snapshot}...")
        for i, sensor in enumerate(globals.SENSORS):
            sensor_name = sensor.value if hasattr(sensor, 'value') else sensor
            ax = axes[i]
            series = sensor_data[sensor_name].dropna()
            if len(series) > 8:
                roll_mean = series.rolling(window=8).mean()
                roll_std = series.rolling(window=8).std()
                ax.plot(series.values, alpha=0.4, label='Raw', color='gray', linestyle=':')
                ax.plot(roll_mean.values, label='Media Mobile', color='blue', linewidth=2)
                ax.plot(roll_std.values, label='Std Mobile', color='red', linewidth=1)
            else:
                ax.text(0.5, 0.5, 'Dati insufficienti', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f"{sensor_name}", fontsize=10)
            ax.grid(True, alpha=0.3)
            if i == 0:
                ax.legend(loc='upper left', fontsize='x-small')
        fig.suptitle(f"Analisi Stazionarietà - ESN {esn_id} | Snapshot: {snapshot}", fontsize=20, y=1.02)
        plt.tight_layout()
        # Salvataggio file con nome specifico per ESN e Snapshot
        filename = f"{esn_id}_Snap_{snapshot}.png".replace("/", "_")
        save_path = os.path.join(output_dir, filename)
        plt.savefig(save_path, bbox_inches='tight')
        plt.close(fig)

In [5]:
results_list = []

for esn_id in globals.ESN:
    sensor_data_tot = train[train['ESN'] == esn_id]    
    for snapshot in globals.Snapshot:
        snapshot_data = sensor_data_tot[sensor_data_tot['Snapshot'] == snapshot]        
        for sensor in globals.SENSORS:
            sensor_name = sensor.value if hasattr(sensor, 'value') else sensor
            series = snapshot_data[sensor_name].dropna()
            # Controllo dei requisiti minimi per ADF
            if series.nunique() <= 1 or len(series) < 10:
                results_list.append({
                    'ESN': esn_id, 
                    'Snapshot': snapshot,
                    'Sensor': sensor_name, 
                    'p-value': np.nan, 
                    'Stazionario': 'Dati Costanti/Insufficienti'
                })
                continue
            try:
                # Esecuzione test ADF
                res = adfuller(series)
                p_value = res[1]
                is_stationary = True if p_value <= 0.05 else False
                results_list.append({
                    'ESN': esn_id, 
                    'Snapshot': snapshot,
                    'Sensor': sensor_name, 
                    'p-value': round(p_value, 4), 
                    'Stazionario': is_stationary
                })
            except Exception as e:
                results_list.append({
                    'ESN': esn_id, 
                    'Snapshot': snapshot,
                    'Sensor': sensor_name, 
                    'p-value': np.nan, 
                    'Stazionario': 'Errore'
                })
df_results = pd.DataFrame(results_list)

# Visualizzazione dei sensori NON stazionari per ogni snapshot
print("Prime 20 righe di sensori NON stazionari (divisi per Snapshot):")
print(df_results[df_results['Stazionario'] == False].head(20))

KeyboardInterrupt: 

In [ ]:
# Matrice di correlazione delle colonne divise per ESN e per ogni snapshot
# Utile per PCA futura
output_dir = globals.PLOT_PATH + "/CORRELATION-MATRIX"
os.makedirs(output_dir, exist_ok=True)
for esn_id in globals.ESN:
    for snapshot in globals.Snapshot:
        cm = train[train["ESN"] == esn_id][train["Snapshot"] == snapshot].corr()
        plt.figure(figsize=(15,15))
        sns.heatmap(cm, annot=True, cmap='coolwarm', fmt=".2f")
        plt.title(f'Correlation Matrix Heatmap - ESN {esn_id} - Snapshot {snapshot}')
        plt.savefig(f"{output_dir}/ESN_{esn_id}_Snapshot_{snapshot}.png".replace("/", "_"))
        plt.close()

In [ ]:
import os
import plotly.express as px

# Define output directory
output_dir = globals.PLOT_PATH + "/CORRELATION-MATRIX-BY-ESN-SNAPSHOT"
os.makedirs(output_dir, exist_ok=True)

for esn_id in globals.ESN:
    for snapshot in globals.Snapshot:
        # Filter data (using & for cleaner boolean indexing)
        subset = train[(train["ESN"] == esn_id) & (train["Snapshot"] == snapshot)]
        
        # Basic check to ensure we have data before plotting
        if subset.empty:
            continue

        cm = subset.corr()

        # Create Plotly Heatmap
        # text_auto=".2f" replaces annot=True and fmt=".2f"
        # 'RdBu_r' is the standard Plotly equivalent to 'coolwarm'
        fig = px.imshow(
            cm,
            text_auto=".2f",
            aspect="auto",
            color_continuous_scale='RdBu_r',
            title=f'Correlation Matrix Heatmap - ESN {esn_id} - Snapshot {snapshot}'
        )

        # Update layout to mimic the large figure size (15x15 inches approx 1200-1500px)
        fig.update_layout(
            width=1200, 
            height=1200,
            title_x=0.5 # Center title
        )

        # Sanitize filename components specifically, rather than the whole path
        safe_esn = str(esn_id).replace("/", "_")
        safe_snap = str(snapshot).replace("/", "_")
        filename_base = f"ESN_{safe_esn}_Snapshot_{safe_snap}"

        # Save as PNG
        fig.write_image(f"{output_dir}/{filename_base}.png")

        # Save as HTML (interactive)
        fig.write_html(f"{output_dir}/{filename_base}.html")


KeyboardInterrupt: 

In [34]:
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Define output directory
output_dir = globals.PLOT_PATH + "/CORRELATION-MATRIX-COMBINED"
os.makedirs(output_dir, exist_ok=True)

# Convert iterables to lists to determine grid dimensions
esn_list = list(globals.ESN)
snap_list = list(globals.Snapshot)
n_rows = len(esn_list)
n_cols = len(snap_list)

# Create a subplot grid
# Rows correspond to ESNs, Columns correspond to Snapshots
fig = make_subplots(
    rows=n_rows, 
    cols=n_cols,
    subplot_titles=[f"ESN {esn} - Snap {snap}" for esn in esn_list for snap in snap_list],
    vertical_spacing=0.04,  # Adjust spacing between rows
    horizontal_spacing=0.04 # Adjust spacing between columns
)
from importlib import reload
reload(globals)
for row_idx, esn_id in enumerate(esn_list):
    for col_idx, snapshot in enumerate(snap_list):
        # Filter data
        subset = train[(train["ESN"] == esn_id) & (train["Snapshot"] == snapshot)]
        subset = globals.sensors_subset(subset)  # Keep only sensor columns
        
        if subset.empty:
            continue

        cm = subset.corr()
        
        # Only show the color scale (legend) on the last plot to reduce clutter
        show_scale = (row_idx == n_rows - 1 and col_idx == n_cols - 1)

        # Add Heatmap Trace
        fig.add_trace(
            go.Heatmap(
                z=cm.values,
                x=cm.columns,
                y=cm.index,
                colorscale='RdBu_r', # Equivalent to 'coolwarm'
                zmin=-1, 
                zmax=1,
                text=cm.values,
                texttemplate="%{z:.2f}", # Mimics annot=True
                textfont={"size": 10},
                showscale=show_scale
            ),
            row=row_idx + 1,
            col=col_idx + 1
        )
        # Invert Y-axis to match standard matrix orientation (top-down)
        fig.update_yaxes(autorange="reversed", row=row_idx + 1, col=col_idx + 1)

# Update layout size dynamically based on the number of plots
# Assuming approx 500px per subplot for readability
fig.update_layout(
    height=1000 * n_rows,
    width=1000 * n_cols,
    title_text="Combined Correlation Matrix Heatmaps"
)

# Save combined files
filename = "Combined_Correlation_Matrices"
fig.write_html(f"{output_dir}/{filename}.html")
fig.write_image(f"{output_dir}/{filename}.png")


In [17]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Define output directory
output_dir = globals.PLOT_PATH + "/CORRELATION-MATRIX"
os.makedirs(output_dir, exist_ok=True)

# Convert iterables to lists
esn_list = list(globals.ESN)
snap_list = list(globals.Snapshot)
n_rows = len(esn_list)
n_cols = len(snap_list)

# Create a grid of subplots
# Adjust figsize: 15 inches per column, 15 inches per row to ensure readability of numbers
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15 * n_cols, 15 * n_rows))

# Ensure axes is always a 2D array for consistent indexing, 
# even if there is only 1 row or 1 column.
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)
elif n_cols == 1:
    axes = axes.reshape(-1, 1)

for i, esn_id in enumerate(esn_list):
    for j, snapshot in enumerate(snap_list):
        ax = axes[i, j]
        
        # Filter data
        subset = train[(train["ESN"] == esn_id) & (train["Snapshot"] == snapshot)]
        
        if subset.empty:
            ax.text(0.5, 0.5, "No Data", ha='center', va='center')
            continue

        cm = subset.corr()

        # Create Heatmap
        sns.heatmap(
            cm, 
            annot=True, 
            cmap='coolwarm', 
            fmt=".2f", 
            ax=ax, 
            cbar=True,
            square=True
        )
        
        ax.set_title(f'ESN {esn_id} - Snapshot {snapshot}')

# Adjust layout to prevent overlap
plt.tight_layout()

# Define filename
filename_base = "Combined_Correlation_Matrices_Matplotlib"
png_path = f"{output_dir}/{filename_base}.png"
html_path = f"{output_dir}/{filename_base}.html"

# 1. Save PNG
plt.savefig(png_path)
plt.close()

# 2. Save HTML (Wrapper for the PNG)
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Combined Correlation Matrices</title>
</head>
<body>
    <h1>Combined Correlation Matrix View</h1>
    <img src="{filename_base}.png" alt="Correlation Matrices" style="max-width: 100%; height: auto;">
</body>
</html>
"""

with open(html_path, "w") as f:
    f.write(html_content)

print(f"Saved combined view to:\n{png_path}\n{html_path}")


Saved combined view to:
./img//CORRELATION-MATRIX/Combined_Correlation_Matrices_Matplotlib.png
./img//CORRELATION-MATRIX/Combined_Correlation_Matrices_Matplotlib.html
